# Setup environment

In [1]:
%reload_ext autoreload
%autoreload 2

**Introduction**

In the real world of machine learning, particularly in fraud detection, disease diagnosis, or anomaly detection, we often face a common challenge: imbalanced datasets. When one class significantly outnumbers the other, our models can become biased, leading to suboptimal performance where the minority class - often the one we're most interested in - gets overlooked.

## Import libraries

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import os
from functools import partial
# import model libraries
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier
from imblearn.under_sampling import RandomUnderSampler
from sklearn.preprocessing import OrdinalEncoder
from imblearn.pipeline import make_pipeline 

import sys
sys.path.append('../../')
from utils.eval_plots import EvalPlots
from utils.model_performance_report import ModelPerformanceReport



#### Define support functions

In [3]:
from collections import defaultdict
from sklearn.base import BaseEstimator

class MultiColumnEncoder(BaseEstimator):
    """https://www.geeksforgeeks.org/label-encoding-across-multiple-columns-in-scikit-learn/ 

    Args:
        BaseEstimator (_type_): _description_
    """
    def __init__(self, columns=None):
        self.columns = columns
        oe = partial(OrdinalEncoder, handle_unknown='use_encoded_value', unknown_value=-1)
        self.encoders = defaultdict(oe)

    def fit(self, X):
        for col in self.columns:
            self.encoders[col].fit(X[[col]])
        return self

    def transform(self, X):
        X_copy = X.copy()  # To avoid modifying the original dataframe
        for col in self.columns:
            X_copy[col] = self.encoders[col].transform(X_copy[[col]])
        return X_copy
    
    def fit_transform(self, X, y=None):
        return self.fit(X).transform(X)

In [4]:
import plotly.express as px
from sklearn.metrics import roc_curve, precision_recall_curve, auc
from plotly.subplots import make_subplots
import plotly.figure_factory as ff

class EvalPlots():

    def __init__(self):
        pass

    def plot_eval_basic(self, y_true, y_score):

        '''
        y_score = model.predict_proba(X)[:, 1]
        '''

        precision, recall, thresholds = precision_recall_curve(y_true, y_score)

        # The histogram of scores compared to true labels
        fig_hist = px.histogram(
            x=y_score, color=y_true, nbins=50,
            labels=dict(color='True Labels', x='Score')
            , histnorm='probability density'
        )

        fig_hist.show()


        # Evaluating model performance on PR curve
        fig_thresh = px.area(
            x=recall, y=precision,
            title=f'Precision-Recall Curve (AUC={auc(recall, precision):.4f})',
            labels=dict(x='Recall', y='Precision'),
            width=700, height=500
        )
        fig_thresh.add_shape(
            type='line', line=dict(dash='dash'),
            x0=0, x1=1, y0=1, y1=0
        )
        fig_thresh.update_yaxes(scaleanchor="x", scaleratio=1)
        fig_thresh.update_xaxes(constrain='domain')

        fig_thresh.show()

        return fig_hist, fig_thresh
    
    
    def plot_eval_pred_dist(self, y_train_true, y_train_pred, y_holdout_true, y_holdout_pred, y_oot_true, y_oot_pred):

        y_df = pd.concat([pd.DataFrame({'y_pred': y_train_pred,
                     'y_true': y_train_true,
                     'dataset': ['train']*len(train_y)}),
            pd.DataFrame({'y_pred': y_holdout_pred,
                     'y_true': y_holdout_true,
                     'dataset': ['holdout']*len(y_holdout_true)}),
            pd.DataFrame({'y_pred': y_oot_pred,
                     'y_true': y_oot_true,
                     'dataset': ['oot']*len(y_oot_true)})]
                )
        fig = px.histogram(y_df,
             x='y_pred', color='y_true', nbins=50,facet_col='dataset',
                    histnorm='probability density',
                    labels=dict(color='True Labels', x='Score'),
                    title='Model Performance')

        return fig
    
    def plot_eval_tpr_fpr_curve(self, y_train_true, y_train_pred, y_holdout_true, y_holdout_pred, y_oot_true, y_oot_pred):
        
        # Evaluating model performance on ROC curve
        fpr, tpr, thresholds = roc_curve(y_train_true, y_train_pred)
        fpr_holdout, tpr_holdout, thresholds_holdout = roc_curve(y_holdout_true, y_holdout_pred)
        fpr_oot, tpr_oot, thresholds_oot = roc_curve(y_oot_true, y_oot_pred)
        fig = make_subplots(rows=1, cols=3, subplot_titles=("Train", "Holdout", "OOT"))
        
        fig_thresh_df = self._generate_tpr_fpr_curve_df(fpr, tpr, thresholds)    
        fig_thresh_holdout_df = self._generate_tpr_fpr_curve_df(fpr_holdout, tpr_holdout, thresholds_holdout)
        fig_thresh_oot_df = self._generate_tpr_fpr_curve_df(fpr_oot, tpr_oot, thresholds_oot)


        fig_thresh = px.line(
            fig_thresh_df, title='TPR and FPR at every threshold',
            width=700, height=500,
            labels=dict(x='Thresholds', y='Values')
        )

        fig_thresh.update_yaxes(scaleanchor="x", scaleratio=1)
        fig_thresh.update_xaxes(range=[0, 1], constrain='domain')


        fig_thresh_holdout = px.line(
            fig_thresh_holdout_df, title='TPR and FPR at every threshold',
            width=700, height=500,
            labels=dict(x='Thresholds', y='Values')
        )

        fig_thresh_holdout.update_yaxes(scaleanchor="x", scaleratio=1)
        fig_thresh_holdout.update_xaxes(range=[0, 1], constrain='domain')
        fig_thresh_holdout.layout.showlegend = False

        fig_thresh_oot = px.line(
            fig_thresh_oot_df, title='TPR and FPR at every threshold',
            width=700, height=500,
            labels=dict(x='Thresholds', y='Values')
        )

        fig_thresh_oot.update_yaxes(scaleanchor="x", scaleratio=1)
        fig_thresh_oot.update_xaxes(range=[0, 1], constrain='domain')
        fig_thresh_oot.layout.showlegend = False

        #add each trace (or traces) to its specific subplot
        pl_nr = 0
        for plot_ in [fig_thresh, fig_thresh_holdout, fig_thresh_oot]:
            pl_nr += 1
            for trace in plot_.data:
                fig.add_trace(trace, row=1, col=pl_nr)
                fig.update_scenes(xaxis_title_text='Thresholds', yaxis_title_text='Values', row=1, col=pl_nr)

        fig.update_layout(title_text="TPR and FPR at every threshold", showlegend=True)


        return fig
        

    def _generate_tpr_fpr_curve_df(self, fpr, tpr, thresholds):
        df = pd.DataFrame({
            'False Positive Rate': fpr,
            'True Positive Rate': tpr
        }, index=thresholds)
        df.index.name = "Thresholds"
        df.columns.name = "Rate"
        df = df.iloc[~df.index.isin([np.inf])]
        df.index = df.index.astype(float)
        return df

    def plot_eval_pr_auc(self, precision_train, recall_train, precision_holdout, recall_holdout, precision_oot, recall_oot):
        # Evaluating model performance on PR curve

        tr_title = f'Train (AUC={auc(recall_train, precision_train):.4f})' 
        ho_title = f'Holdout (AUC={auc(recall_holdout, precision_holdout):.4f})'
        oot_title = f'OOT (AUC={auc(recall_oot, precision_oot):.4f})'

        fig = make_subplots(rows=1, cols=3, subplot_titles=(tr_title, ho_title, oot_title)) 

        trace0 = px.area(
            x=recall_train, y=precision_train,
            title=f'Training (AUC={auc(recall_train, precision_train):.4f})',
            labels=dict(x='Recall', y='Precision'),
            width=700, height=500
        )
        trace0.add_shape(
            type='line', line=dict(dash='dash'),
            x0=0, x1=1, y0=1, y1=0
        )
        trace0.update_yaxes(scaleanchor="x", scaleratio=1)
        trace0.update_xaxes(constrain='domain')

        trace1 = px.area(
            x=recall_holdout, y=precision_holdout,
            title=f'Holdout (AUC={auc(recall_holdout, precision_holdout):.4f})',
            labels=dict(x='Recall', y='Precision'),
            width=700, height=500
        )
        trace1.add_shape(
            type='line', line=dict(dash='dash'),
            x0=0, x1=1, y0=1, y1=0
        )
        trace1.update_yaxes(scaleanchor="x", scaleratio=1)
        trace1.update_xaxes(constrain='domain')

        trace2 = px.area(
            x=recall_oot, y=precision_oot,
            title=f'OOT AUC={auc(recall_oot, precision_oot):.4f})',
            labels=dict(x='Recall', y='Precision'),
            width=700, height=500
        )
        trace2.add_shape(
            type='line', line=dict(dash='dash'),
            x0=0, x1=1, y0=1, y1=0
        )
        trace2.update_yaxes(scaleanchor="x", scaleratio=1)
        trace2.update_xaxes(constrain='domain')

        pl_nr = 0
        for plot_ in [trace0, trace1, trace2]:
            pl_nr += 1
            for trace in plot_.data:
                fig.add_trace(trace, row=1, col=pl_nr)

        fig.update_layout(title_text="Model Precision-Recall Curve", showlegend=True)

        fig.update_layout(scene=dict(
            yaxis_title = 'Precision',
            xaxis_title = 'Recall'),
            scene2 = dict(
                yaxis_title = 'Precision',
                xaxis_title = 'Recall'),
            scene3 = dict(
                yaxis_title = 'Precision',
                xaxis_title = 'Recall')
        )


        return fig
    

    def plot_eval_roc_auc(self, y_train_true, y_train_pred, y_holdout_true, y_holdout_pred, y_oot_true, y_oot_pred):
        # Evaluating model performance on ROC curve
        fpr, tpr, _ = roc_curve(y_train_true, y_train_pred)
        fpr_holdout, tpr_holdout, _ = roc_curve(y_holdout_true, y_holdout_pred)
        fpr_oot, tpr_oot, _ = roc_curve(y_oot_true, y_oot_pred)
        
        title_train = f'ROC Curve (AUC={auc(fpr, tpr):.4f})'
        title_holdout = f'ROC Curve (AUC={auc(fpr_holdout, tpr_holdout):.4f})'
        title_oot = f'ROC Curve (AUC={auc(fpr_oot, tpr_oot):.4f})'

        fig = make_subplots(rows=1, cols=3, subplot_titles=(title_train, title_holdout, title_oot))  

        roc_train = px.area(
                x=fpr, y=tpr,
                title=title_train,
                labels=dict(x='False Positive Rate', y='True Positive Rate'),
                width=700, height=500
            )
        roc_train.add_shape(
            type='line', line=dict(dash='dash'),
            x0=0, x1=1, y0=0, y1=1
        )

        roc_train.update_yaxes(scaleanchor="x", scaleratio=1)
        roc_train.update_xaxes(constrain='domain')
        
        roc_holdout = px.area(
                x=fpr_holdout, y=tpr_holdout,
                title=title_holdout,
                labels=dict(x='False Positive Rate', y='True Positive Rate'),
                width=700, height=500
            )
        roc_holdout.add_shape(
            type='line', line=dict(dash='dash'),
            x0=0, x1=1, y0=0, y1=1
        )

        roc_holdout.update_yaxes(scaleanchor="x", scaleratio=1)
        roc_holdout.update_xaxes(constrain='domain')

        roc_oot = px.area(
                x=fpr_oot, y=tpr_oot,
                title=title_oot,
                labels=dict(x='False Positive Rate', y='True Positive Rate'),
                width=700, height=500
            )
        roc_oot.add_shape(
            type='line', line=dict(dash='dash'),
            x0=0, x1=1, y0=0, y1=1
        )

        roc_oot.update_yaxes(scaleanchor="x", scaleratio=1)
        roc_oot.update_xaxes(constrain='domain')

        pl_nr = 0
        for plot_ in [roc_train, roc_holdout, roc_oot]:
            pl_nr += 1
            for trace in plot_.data:
                fig.add_trace(trace, row=1, col=pl_nr)

        fig.update_layout(title_text="ROC Curve", showlegend=True)
        fig.add_shape(
            type='line', line=dict(dash='dash'),
            x0=0, x1=1, y0=0, y1=1
        )
        return fig
    


In [5]:
## Build model performance report
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix


class ModelPerformanceReport(EvalPlots):
    def __init__(self, train_X,train_y, holdout_X,holdout_y,oot_X, oot_y):
        self.train_X = train_X
        self.train_y = train_y
        self.holdout_X = holdout_X
        self.holdout_y = holdout_y
        self.oot_X = oot_X
        self.oot_y = oot_y
        super().__init__()

    def predictions(self, model):
        y_train_pred = model.predict(self.train_X)
        y_train_true = self.train_y
        y_holdout_pred = model.predict(self.holdout_X[self.train_X.columns])
        y_holdout_true = self.holdout_y
        y_oot_true = self.oot_y
        y_oot_pred = model.predict(self.oot_X[self.train_X.columns])

        return y_train_pred, y_train_true, y_holdout_pred, y_holdout_true, y_oot_pred, y_oot_true

    def produce_report(self, model): 
        
        y_train_pred, y_train_true, y_holdout_pred, y_holdout_true, y_oot_pred, y_oot_true = self.predictions(model)

        # confusion_matrix(y_train_true, y_train_pred)
        results_df = pd.DataFrame()
        results_df['train'] = [accuracy_score(y_train_true, y_train_pred), precision_score(y_train_true, y_train_pred), recall_score(y_train_true, y_train_pred), f1_score(y_train_true, y_train_pred)]
        results_df['holdout'] = [accuracy_score(y_holdout_true, y_holdout_pred), precision_score(y_holdout_true, y_holdout_pred), recall_score(y_holdout_true, y_holdout_pred), f1_score(y_holdout_true, y_holdout_pred)]   
        results_df['oot'] = [accuracy_score(y_oot_true, y_oot_pred), precision_score(y_oot_true, y_oot_pred), recall_score(y_oot_true, y_oot_pred), f1_score(y_oot_true, y_oot_pred)]
        results_df.index = ['accuracy', 'precision', 'recall', 'f1']
        return results_df
    

    def proba_predictions(self, model):
        y_train_pred = model.predict_proba(self.train_X)[:, 1]
        y_train_true = self.train_y
        y_holdout_pred = model.predict_proba(self.holdout_X[self.train_X.columns])[:, 1]
        y_holdout_true = self.holdout_y
        y_oot_true = self.oot_y
        y_oot_pred = model.predict_proba(self.oot_X[self.train_X.columns])[:, 1]

        return y_train_pred, y_train_true, y_holdout_pred, y_holdout_true, y_oot_pred, y_oot_true
    
    def produce_proba_report(self, model):
        y_train_pred, y_train_true, y_holdout_pred, y_holdout_true, y_oot_pred, y_oot_true= self.proba_predictions(model)
        return self.plot_eval_pred_dist(y_train_true, y_train_pred, y_holdout_true, y_holdout_pred, y_oot_true, y_oot_pred)

    def precision_recall_calc(self, y_train_true, y_train_pred, y_holdout_true, y_holdout_pred, y_oot_true, y_oot_pred):
        precision_train, recall_train, _ = precision_recall_curve(y_train_true, y_train_pred)
        precision_holdout, recall_holdout, _ = precision_recall_curve(y_holdout_true, y_holdout_pred)
        precision_oot, recall_oot, _ = precision_recall_curve(y_oot_true, y_oot_pred)
        return precision_train, recall_train, precision_holdout, recall_holdout, precision_oot, recall_oot

    def produce_pr_auc_report(self, model):
        y_train_pred, y_train_true, y_holdout_pred, y_holdout_true, y_oot_pred, y_oot_true= self.proba_predictions(model)
        precision_train, recall_train, precision_holdout, recall_holdout, precision_oot, recall_oot = self.precision_recall_calc(y_train_true, y_train_pred, y_holdout_true, y_holdout_pred, y_oot_true, y_oot_pred)
        return self.plot_eval_pr_auc(precision_train, recall_train, precision_holdout, recall_holdout, precision_oot, recall_oot) 
    
    def plot_eval_roc_auc_report(self, model):
        y_train_pred, y_train_true, y_holdout_pred, y_holdout_true, y_oot_pred, y_oot_true = self.proba_predictions(model)
        return self.plot_eval_roc_auc(y_train_true, y_train_pred, y_holdout_true, y_holdout_pred, y_oot_true, y_oot_pred)


In [6]:
class ModelPipeline():
    def __init__(self, data, metadata_columns) -> None:
        self.data = data
        self.metadata_columns = metadata_columns
        self.oot_X = pd.DataFrame()
        self.oot_y = pd.Series()
        self.holdout_X = pd.DataFrame()
        self.holdout_y = pd.Series()
        self.train_X = pd.DataFrame()
        self.train_y = pd.Series()

    def prepare_dataset(self):
        _expression = "trans_date_trans_time < '2020-07-01 00:00:00'"
        print(f"Splitting dataframe based on expression {_expression!r}.")
        self.data.index = self.data.trans_num
        train = self.data.query(_expression)
        oot = self.data.query(f"~({_expression})")
        print(f"Split dataframe into two dataframes with shapes {train.shape} and {oot.shape}.")

        oot_y = oot.is_fraud
        oot_X = oot.drop(columns=['is_fraud'])
        self.oot_y = oot_y
        self.oot_X = oot_X

        # train test split
        X = train.drop(columns=['is_fraud'])
        X.drop(self.metadata_columns, axis=1, inplace=True)
        y = train['is_fraud']

        train_X, holdout_X, train_y, holdout_y = train_test_split(X, y, test_size=0.2, random_state=42)
        train_X.drop(columns=['age_at_purchase'], inplace=True)
        holdout_X.drop(columns=['age_at_purchase'], inplace=True)

        holdout_y = train.is_fraud
        holdout_X = train.drop(columns=['is_fraud'])
        self.holdout_y = holdout_y
        self.holdout_X = holdout_X
        self.train_X = train_X
        self.train_y = train_y

    def encode_categorical_variables(self):
        categorical_columns = self.train_X.select_dtypes(include=['object']).columns.tolist()
        encoder = MultiColumnEncoder(categorical_columns)
        self.train_X = encoder.fit_transform(self.train_X)
        self.holdout_X = encoder.transform(self.holdout_X)
        self.oot_X = encoder.transform(self.oot_X)

    def run_model(self):
        undersample_pipe = make_pipeline(RandomUnderSampler(sampling_strategy=0.2, random_state=42)
                                    ,LGBMClassifier(objective='binary'))
        self.score_balanced_model = undersample_pipe.fit(self.train_X, self.train_y
                                            , lgbmclassifier__eval_metric='average_precision'
                                            )
        
    def main(self):
        self.prepare_dataset()
        self.encode_categorical_variables()
        self.run_model()
        return self.score_balanced_model, self.train_X, self.train_y, self.holdout_X, self.holdout_y, self.oot_X, self.oot_y

    

## import the dataset

In [7]:
data = pd.read_parquet('data/credit_card_transactions.parquet')

In [8]:
metadata_columns = ['trans_date_trans_time','gender','street','trans_num']

## Run a basic model with downsampling on dataset

In [ ]:
mp = ModelPipeline(data, metadata_columns)
score_balanced_model, train_X, train_y, holdout_X, holdout_y, oot_X, oot_y = mp.main()

In [10]:
#model dump
import pickle
with open('models/score_balanced_model.pkl', 'wb') as f:
    pickle.dump(score_balanced_model, f)

In [11]:
#dump datasets
with open('models/train_X.pkl', 'wb') as f:
    pickle.dump(train_X, f)
with open('models/train_y.pkl', 'wb') as f:
    pickle.dump(train_y, f)
with open('models/holdout_X.pkl', 'wb') as f:
    pickle.dump(holdout_X, f)
with open('models/holdout_y.pkl', 'wb') as f:
    pickle.dump(holdout_y, f)
with open('models/oot_X.pkl', 'wb') as f:
    pickle.dump(oot_X, f)
with open('models/oot_y.pkl', 'wb') as f:
    pickle.dump(oot_y, f)

## Let's look at the metrics

confusion matrix metrics:
- precision
- recall
- accuracy
- F1

PR-AUC
ROC-AUC

In [12]:
report_class = ModelPerformanceReport(train_X,train_y,holdout_X,holdout_y,oot_X,oot_y)
eval_plots = EvalPlots()

Lets score the datasets

In [13]:
y_train_pred, y_train_true, y_holdout_pred, y_holdout_true, y_oot_true, y_oot_pred = report_class.predictions(score_balanced_model)

## METRICS

### CONFUSION MATRIX

<img src="/images/metrics/enhanced_confusion_matrix.png">

__metrics__
* Accuracy: 
    - Answers to: How often is the classifier correct?
    - Ratio of correct predictions to total predictions: $Accuracy = \frac{TP + TN}{TP + TN + FP + FN}$
    
* Precision: 
    - Answers: When it predicts positive, how often is it right? 
    - Ratio of correct positive predictions to total positive predictions: $Precision = \frac{TP}{TP + FP}$

* Recall: 
    - Answers: When it's actually positive, how often does it predict it?
    - Ratio of correct positive predictions to total actual positives: $Recall = \frac{TP}{TP + FN}$

* F1 Score: 
    - Answers: 
    - Harmonic mean of precision and recall: $F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall}$


Lets look at an example. High accuracy might seems good, but with such unbalanced dataset high accuracy is easily obtained by scoring as 0 most of the population

In [ ]:
# metrics for the different datasets
report_class.produce_report(score_balanced_model)

In [ ]:
#Lets look at the distribution of the predictions
df = pd.DataFrame(list(map(list, zip(*[ pd.Series(y_oot_pred).value_counts( normalize=True).tolist()
                , pd.Series(y_oot_true).value_counts( normalize=True).tolist(),
                pd.Series(y_holdout_pred).value_counts( normalize=True).tolist()
                , pd.Series(y_holdout_true).value_counts( normalize=True).tolist(),
                pd.Series(y_train_pred).value_counts( normalize=True).tolist()
                ,  pd.Series(y_train_true).value_counts( normalize=True).tolist()
                ]))),
              index=pd.Index(['Non Fraud (Negative)', 'Fraud (Positive)'], name='Labels'),
              columns=pd.MultiIndex.from_product([['OOT', 'Holdout', 'Train'],['Pred', 'True']], names=['Dataset:', 'y'])
              )
df

In [ ]:
df

### PR-AUC vs ROC-AUC

**General Notions**
The PR-AUC plots provide a more realistic picture of model performance in fraud detection scenarios, as they better capture the challenges of detecting rare fraudulent transactions while maintaining reasonable precision. This is particularly important in fraud detection where both false positives (blocking legitimate transactions) and false negatives (missing fraud) have significant business implications.
In details:

ROC AUC plots can be misleading in highly imbalanced datasets because:
* They show the trade-off between True Positive Rate (TPR) and False Positive Rate (FPR): a model can achieve high ROC AUC by simply predicting the majority class
* The ROC curve might look good even when the model is not performing well on the minority class
Therefore, ROC plot is less sensitive to class imvalance and might lead to optimistic considerations. In addition, it gives no insight on the capability of the model to detect rare event.

PR AUC plots are more informative because:
* They show the trade-off between Precision and Recall, therefore focusing on the positive class (fraudulent transactions).
* They better reflect the model's ability to handle the minority class, providing a clearer picture of the model's practical utility


__Commenting our Plots__

Looking at the model performance across different datasets, the PR-AUC shows that the model is overfitting, while the ROC-AUC gives a stable very high performance (around 0.99).
The significant 0.4 drop of PR from Training to Out-of-Time (OOT) indicates poor generalization to new data and a struggle to maintain precision while keeping recall high. All indicators of overfitting.


**Best Practices for Fraud Detection:**

* Use PR-AUC as the primary evaluation metric
* Monitor both precision and recall trade-offs
* Consider the business impact of false positives vs false negatives
* Implement proper sampling techniques within cross-validation folds
* Use stratified cross-validation to maintain class distribution




In [ ]:
report_class.produce_pr_auc_report(score_balanced_model)

In [ ]:
report_class.plot_eval_roc_auc_report(score_balanced_model)

In [ ]:
y_train_pred, y_train_true, y_holdout_pred, y_holdout_true, y_oot_pred, y_oot_true  =   report_class.proba_predictions(score_balanced_model)
report_class.plot_eval_tpr_fpr_curve(y_train_true, y_train_pred, y_holdout_true, y_holdout_pred, y_oot_true, y_oot_pred)

In [ ]:
report_class.produce_proba_report(score_balanced_model)

### Lets see the effects of a lower fraud rate (fraud population varies)

In [ ]:
from fraud_rate_perturbation_simulation import create_altered_datasets, plot_curves, plot_prediction_distribution

datasets = create_altered_datasets(oot_X[train_X.columns], oot_y, fraud_rates=[0.001, 0.005, 0.01, 0.02, 0.05])
plot_curves(datasets, score_balanced_model)

### Lets now compare PR and ROC for population drifting datasets

This analysis helps us understand:
* How well the model performs when fraud patterns change
* Whether the model is more sensitive to certain types of shifts
* The model's robustness to different types of fraud pattern changes

Following, we generate 4 shifts of the fraudulent featue distribution and then test PR and ROC curves on each new dataset.

Details on shifts:

1. Original (No Shift) - shift_factor = 0
    * This is the baseline dataset with no modifications
    * Used as a reference point to compare against other shifts
2. Positive Shift - shift_factor = 1
    * All numeric features for fraud cases are shifted upward
    * For each numeric column, fraud cases are increased by 1 standard deviation
    * This simulates fraud cases becoming more extreme in the positive direction
    * Example: If a feature has values [1, 2, 3] for fraud cases, they become [2, 3, 4]

3. Negative Shift - shift_factor = -1
    * All numeric features for fraud cases are shifted downward
    * For each numeric column, fraud cases are decreased by 1 standard deviation
    * This simulates fraud cases becoming more extreme in the negative direction
    * Example: If a feature has values [1, 2, 3] for fraud case

4. Mixed Shift - shift_factor = 0.5
    * Features for fraud cases are randomly shifted in both directions
    * For each numeric column:
        - Randomly decides whether to shift up or down
        - Uses np.random.choice([-1, 1]) to determine direction
        - Applies a 0.5 standard deviation shift in the chosen direction
    * This simulates fraud cases becoming more extreme but in different directions for different features
    * Example: If we have two features:
        - Feature 1: [1, 2, 3] might become [1.5, 2.5, 3.5] (shifted up)
        - Feature 2: [1, 2, 3] might become [0.5, 1.5, 2.5] (shifted down)

Important notes about all shifts:
* Only fraud cases (where y == 1) are modified
* Non-fraud cases remain unchanged
* After shifting, all numeric features are standardized using StandardScaler
* The fraud rate remains the same across all shifts
* The shifts help us understand how robust the model is to changes in the fraud population's characteristics

In [ ]:
from fraud_population_shift import create_shifted_datasets, plot_curves, plot_prediction_distribution

datasets = create_shifted_datasets(oot_X[train_X.columns], oot_y)
plot_curves(datasets, score_balanced_model)

In [47]:
#plot_prediction_distribution(datasets, score_balanced_model)